# Part 2.10 Appendix: Sandboxes

After this appendix you know when an agent needs a sandbox and how one plugs into the
backend you built in [2.4](04_backends.ipynb). This notebook is reading only. Nothing here
runs.

## See it

Our agent only reads and writes files. If an agent needs to run code, give it a
**sandbox**: an isolated environment where it can run shell commands. Generated code can't
reach your credentials, your files, or your network.

With a sandbox backend the agent gets everything it had before, plus one more tool:

| Tool | Comes from |
|---|---|
| `ls`, `read_file`, `write_file`, `edit_file`, `glob`, `grep` | any backend |
| `execute` (run a shell command) | a sandbox backend only |

Typical uses: coding agents that run tests, and data-analysis agents that install
libraries and make charts.

## Build it

A sandbox is a backend, so you pass it as `backend=` like every backend before it. To
keep the agent's memory across threads, make the sandbox the `default` of a
`CompositeBackend` and keep the other routes. The `execute` tool goes to the default
backend.

```python
from deepagents import create_deep_agent
from deepagents.backends import CompositeBackend, LangSmithSandbox, StoreBackend
from langsmith.sandbox import SandboxClient

client = SandboxClient()
sandbox = client.create_sandbox()

backend = CompositeBackend(
    default=LangSmithSandbox(sandbox=sandbox),
    routes={"/memories/": StoreBackend(namespace=lambda rt: ("kea-air", "memories"))},
)

agent = create_deep_agent(model=MODEL, backend=backend)
try:
    agent.invoke({"messages": [("user", "Write a script that sums these numbers and run it")]})
finally:
    client.delete_sandbox(sandbox.name)
```

Always delete the sandbox when you are done, as the `finally` does.

## Under the hood

- **Providers.** LangSmith offers managed sandboxes. Daytona and others have integrations
  too. They all implement the same sandbox backend protocol.
- **Not the same as `LocalShellBackend`.** That one runs commands on your own machine with
  your permissions. It has no isolation, so use it only in controlled development
  environments.
- **Secrets.** Keep credentials out of the sandbox. Anything the agent can read there, a
  prompt injection can read too.
- **Approval still applies.** `interrupt_on={"execute": True}` pauses before each command,
  the same way it paused `issue_credit`.

Docs: [Sandboxes](https://docs.langchain.com/oss/python/deepagents/sandboxes),
[Backends](https://docs.langchain.com/oss/python/deepagents/backends).

## Recap

- A sandbox is a backend that adds an `execute` tool inside an isolated environment.
- Plug it into a `CompositeBackend` as the `default` and keep your memory routes.
- Delete it when you are done, and keep secrets out of it.

**Next:** [Part 3.1 The deployed agent](../part3-deploy-monitor/01_deployed_agent.ipynb)